In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import pandas as pd
import numpy as np
import time


In [4]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

df = pd.read_csv(url)

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


In [6]:
df.isnull().sum()

,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [11]:
df["age"] = df["age"].fillna(df["age"].median())

In [12]:
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

In [13]:
df["sex"] = df["sex"].map({
    "male": 0,
    "female": 1
})

In [14]:
df["embarked"] = df["embarked"].map({
    "S": 0,
    "C": 1,
    "Q": 2
})

In [15]:
df[["sex", "embarked"]].head()

,sex,embarked
0,0,0
1,1,1
2,1,0
3,1,0
4,0,0


In [16]:
df["family_size"] = df["sibsp"] + df["parch"] + 1

In [17]:
df["fare_person"] = df["fare"] / df["family_size"]

In [18]:
df[["sibsp", "parch", "family_size", "fare", "fare_person"]].head()

,sibsp,parch,family_size,fare,fare_person
0,1,0,2,7.2500,3.62500
1,1,0,2,71.2833,35.64165
2,0,0,1,7.9250,7.92500
3,1,0,2,53.1000,26.55000
4,0,0,1,8.0500,8.05000


In [19]:
final_df = df.drop(
    columns=[
        "class",
        "who",
        "adult_male",
        "deck",
        "embark_town",
        "alive",
        "alone"
    ]
)

final_df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,family_size,fare_person
0,0,3,0,22.0,1,0,7.2500,0,2,3.62500
1,1,1,1,38.0,1,0,71.2833,1,2,35.64165
2,1,3,1,26.0,0,0,7.9250,0,1,7.92500
3,1,1,1,35.0,1,0,53.1000,0,2,26.55000
4,0,3,0,35.0,0,0,8.0500,0,1,8.05000


In [20]:
X = final_df.drop("survived", axis=1)
y = final_df["survived"]

In [21]:
X.head()

,pclass,sex,age,sibsp,parch,fare,embarked,family_size,fare_person
0,3,0,22.0,1,0,7.2500,0,2,3.62500
1,1,1,38.0,1,0,71.2833,1,2,35.64165
2,3,1,26.0,0,0,7.9250,0,1,7.92500
3,1,1,35.0,1,0,53.1000,0,2,26.55000
4,3,0,35.0,0,0,8.0500,0,1,8.05000


In [22]:
y.head()

,survived
0,0
1,1
2,1
3,1
4,0


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(712, 9)
(179, 9)
(712,)
(179,)


In [25]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [26]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

(712, 9)
(179, 9)


In [28]:
models = {
    "의사결정나무": DecisionTreeClassifier(random_state=42),

    "랜덤 포리스트": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "로지스틱 회귀": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "나이브 베이즈": GaussianNB(),

    "KNN": KNeighborsClassifier(),

    "Linear SVM": LinearSVC(
        max_iter=10000,
        random_state=42
    ),

    "퍼셉트론": Perceptron(
        random_state=42
    ),

    "SGD": SGDClassifier(
        random_state=42
    )
}

In [29]:
models

{'의사결정나무': DecisionTreeClassifier(random_state=42),
 '랜덤 포리스트': RandomForestClassifier(random_state=42),
 '로지스틱 회귀': LogisticRegression(max_iter=1000, random_state=42),
 '나이브 베이즈': GaussianNB(),
 'KNN': KNeighborsClassifier(),
 'Linear SVM': LinearSVC(max_iter=10000, random_state=42),
 '퍼셉트론': Perceptron(random_state=42),
 'SGD': SGDClassifier(random_state=42)}

In [30]:
results = []

for name, model in models.items():

    start_time = time.time()

    model.fit(X_train_scaled, y_train)

    predictions = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, predictions)

    elapsed_time = time.time() - start_time

    results.append({
        "모델": name,
        "정확도": accuracy,
        "처리시간": elapsed_time
    })

In [31]:
result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    by="정확도",
    ascending=False
)

result_df

,모델,정확도,처리시간
2,로지스틱 회귀,0.798883,0.018771
5,Linear SVM,0.798883,0.003993
4,KNN,0.793296,0.009210
1,랜덤 포리스트,0.793296,0.233894
3,나이브 베이즈,0.770950,0.002879
0,의사결정나무,0.770950,0.009529
7,SGD,0.748603,0.007192
6,퍼셉트론,0.614525,0.003992


In [32]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

cv_scores = cross_val_score(
    random_forest,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("각 교차검증 정확도:", cv_scores)
print("평균 정확도:", cv_scores.mean())

각 교차검증 정확도: [0.78212291 0.79213483 0.85393258 0.78651685 0.80898876]
평균 정확도: 0.8047391877471597




---



랜덤 포리스트 모델을 5겹 교차검증한 결과,
각 정확도는 약 78.2%, 79.2%, 85.4%, 78.7%, 80.9%로 나타났다.

평균 정확도는 약 80.47%로,
랜덤 포리스트 모델이 타이타닉 승객의 생존 여부를
평균적으로 약 80% 정확도로 예측했다.